In [1]:
# graphrag_plus.py

import json
import pandas as pd
import networkx as nx

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

import ollama



/Users/iramkamdar/miniconda3/envs/genai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load FAQ (already generated from refined.csv using your script)
faq = pd.read_csv("faq.csv", encoding="utf-8")

# Load merged JSON with email content (prospect_email and reply)
with open("student_email_pairs_merged.json", "r", encoding="utf-8") as f:
    labels = json.load(f)

print(f"FAQ rows: {len(faq)}")
print(f"Labelled items: {len(labels)}")
print(f"Sample item keys: {list(labels[0].keys()) if labels else 'No data'}")


FAQ rows: 13
Labelled items: 111
Sample item keys: ['id', 'labels', 'subject', 'sender_email', 'prospect_email', 'reply']


In [3]:
# Extract all unique intents from merged data
unique_intents = set()
for item in labels:
    for intent in item.get("labels", {}).get("intents", []):
        unique_intents.add(intent)

print("Unique Intents:", unique_intents)


Unique Intents: {'send_materials', 'request_info', 'accept_or_decline', 'request_feedback', 'share_feedback', 'reschedule', 'follow_up', 'schedule', 'confirm'}


In [4]:
# Preview merged data structure
for i, item in enumerate(labels[:3]):
    print(f"\n=== Item {i+1} ===")
    print(f"ID: {item.get('id')}")
    print(f"Subject: {item.get('subject')}")
    print(f"Has prospect_email: {'prospect_email' in item and item['prospect_email'] is not None}")
    print(f"Has reply: {'reply' in item and item['reply'] is not None}")
    print(f"Labels: {item.get('labels')}")
    if item.get('prospect_email'):
        print(f"Prospect email preview: {item['prospect_email'][:100]}...")



=== Item 1 ===
ID: a1bc948c-4475-4b2f-ada8-84de14a52a57
Subject: Offer for RA Position in Our Lab
Has prospect_email: True
Has reply: True
Labels: {'topic': 'Professor/Academic', 'intents': ['accept_or_decline', 'share_feedback'], 'artifacts': ['offer_letter']}
Prospect email preview: Hi Zubair,

I hope this message finds you well. Im delighted to inform you that your application fo...

=== Item 2 ===
ID: 406b8fae-4c32-4b7f-bfdf-ba5642bdd466
Subject: Team Meeting Availability Confirmation
Has prospect_email: True
Has reply: True
Labels: {'topic': 'Group/Event Coordination', 'intents': ['accept_or_decline', 'reschedule', 'request_info'], 'artifacts': []}
Prospect email preview: Hi Zubair,

I hope this email finds you well. Were planning a team meeting for our project next wee...

=== Item 3 ===
ID: 9180c14f-f779-4339-8495-eb765e329618
Subject: Discussion about Recent Submission
Has prospect_email: True
Has reply: True
Labels: {'topic': 'Professor/Academic', 'intents': ['share_feedbac

In [5]:
import networkx as nx

G = nx.DiGraph()

# Store email content mapping for each node
# This will help us include personalized email context in embeddings
email_content_map = {}  # {node_name: [list of email contexts]}

for item in labels:
    label_data = item.get("labels", {})
    topic = label_data.get("topic")
    intents = label_data.get("intents", [])
    artifacts = label_data.get("artifacts", [])
    
    # Get email content for personalization
    prospect_email = item.get("prospect_email", "")
    reply = item.get("reply", "")
    subject = item.get("subject", "")
    email_id = item.get("id", "")
    
    # Create email context string
    email_context = f"Subject: {subject}\nEmail: {prospect_email[:200] if prospect_email else 'N/A'}\nReply: {reply[:200] if reply else 'N/A'}"

    if not topic and not intents and not artifacts:
        continue

    # Add topic node with email context
    if topic:
        G.add_node(topic, type="topic")
        if topic not in email_content_map:
            email_content_map[topic] = []
        email_content_map[topic].append({
            "email_id": email_id,
            "subject": subject,
            "prospect_email": prospect_email[:300] if prospect_email else "",
            "reply": reply[:300] if reply else ""
        })

    # Add intents as nodes and edges with email context
    for intent in intents:
        G.add_node(intent, type="intent")
        if topic:
            G.add_edge(topic, intent, relation="HAS_INTENT", email_id=email_id)
        
        # Store email context for intent nodes
        if intent not in email_content_map:
            email_content_map[intent] = []
        email_content_map[intent].append({
            "email_id": email_id,
            "subject": subject,
            "prospect_email": prospect_email[:300] if prospect_email else "",
            "reply": reply[:300] if reply else ""
        })

    # Add artifacts as nodes and edges with email context
    for artifact in artifacts:
        G.add_node(artifact, type="artifact")
        if topic:
            G.add_edge(topic, artifact, relation="USES_ARTIFACT", email_id=email_id)
        
        # Store email context for artifact nodes
        if artifact not in email_content_map:
            email_content_map[artifact] = []
        email_content_map[artifact].append({
            "email_id": email_id,
            "subject": subject,
            "prospect_email": prospect_email[:300] if prospect_email else "",
            "reply": reply[:300] if reply else ""
        })

print(f"Graph: {len(G.nodes())} nodes, {len(G.edges())} edges")
print(f"Nodes with email context: {len([n for n in G.nodes() if n in email_content_map])}")
print(f"Sample node with context: {list(email_content_map.keys())[0] if email_content_map else 'None'}")


Graph: 31 nodes, 99 edges
Nodes with email context: 31
Sample node with context: Professor/Academic


In [6]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

qdrant = QdrantClient(path="qdrant_data")   # persistent local storage

qdrant.recreate_collection(
    collection_name="knowledge_space",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)


True

## if the above cell doesn't work , RUN rm -f qdrant_data/.lock      

In [7]:
faq_texts = [
    f"FAQ | Question: {row['question']} | Answer: {row['answer']}"
    for _, row in faq.iterrows()
]
faq_vectors = embedder.encode(faq_texts, show_progress_bar=False)
faq_payloads = []

for idx, row in faq.iterrows():
    faq_payloads.append({
        "type": "faq",
        "id_kind": "faq",
        "faq_id": int(idx),
        "question": row["question"], 
        "answer": row["answer"],
    })


In [8]:
graph_nodes = list(G.nodes(data=True))  # [(name, attrs), ...]

graph_texts = []
graph_payloads = []

for i, (name, attrs) in enumerate(graph_nodes):
    ntype = attrs.get("type", "unknown")
    neighbors = list(G.successors(name)) + list(G.predecessors(name))
    neighbors_str = ", ".join(neighbors) if neighbors else "None"
    
    # Get email context for this node (personalized content)
    email_contexts = email_content_map.get(name, [])
    
    # Build enriched text with email content for better personalization
    email_examples = ""
    if email_contexts:
        # Include up to 2 email examples for context
        for ctx in email_contexts[:2]:
            if ctx.get("prospect_email"):
                email_examples += f"\nExample Email: {ctx['prospect_email'][:150]}"
            if ctx.get("reply"):
                email_examples += f"\nExample Reply: {ctx['reply'][:150]}"
    
    # Enhanced text with email context
    text = f"GRAPH_NODE | Type: {ntype} | Name: {name} | Neighbors: {neighbors_str}{email_examples}"
    graph_texts.append(text)

    # Enhanced payload with email content
    graph_payloads.append({
        "type": "graph_node",
        "id_kind": "graph_node",
        "node_name": name,
        "node_type": ntype,
        "neighbors": neighbors,
        "email_contexts": email_contexts[:3],  # Store up to 3 email examples
        "email_count": len(email_contexts),  # Total number of emails associated
    })

graph_vectors = embedder.encode(graph_texts, show_progress_bar=False)
print(f"✅ Encoded {len(graph_vectors)} graph nodes with personalized email content")


✅ Encoded 31 graph nodes with personalized email content


In [9]:
points = []

# FAQ points
for i, (vec, payload) in enumerate(zip(faq_vectors, faq_payloads)):
    points.append(
        PointStruct(
            id=i,
            vector=vec.tolist(),
            payload=payload
        )
    )

offset = len(points)

# Graph-node points (id continues after FAQ)
for j, (vec, payload) in enumerate(zip(graph_vectors, graph_payloads)):
    points.append(
        PointStruct(
            id=offset + j,
            vector=vec.tolist(),
            payload=payload
        )
    )

qdrant.upsert(collection_name="knowledge_space", points=points)
print(f"✅ Upserted {len(points)} total points into 'knowledge_space'")


✅ Upserted 44 total points into 'knowledge_space'


In [1]:
# Close the Qdrant client to release the lock
qdrant = None
import gc
gc.collect()

31

In [3]:
# Release Qdrant connection
qdrant = None
import gc
gc.collect()
print("✅ Qdrant released - backend can now start")

✅ Qdrant released - backend can now start


In [15]:
import ollama
import json

def classify_intent_llm(email_text: str, available_intents: list):
    """
    Use LLM (Ollama) to classify the intent based on email text and your labeled intent list.
    """
    prompt = f"""
    You are an intent classification assistant.
    Below is a list of valid intents extracted from training data:
    {available_intents}

    Read the email carefully and return ONLY a JSON object with the most relevant intent.
    If multiple intents fit, return the one that best describes the user's main goal.

    Email:
    \"\"\"{email_text}\"\"\"

    Output example:
    {{"intent": "request_feedback"}}
    """

    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )

    try:
        content = response["message"]["content"]
        parsed = json.loads(content)
        return parsed.get("intent", "general_inquiry")
    except Exception:
        return "general_inquiry"


In [10]:
def build_prompt(email_text, intent, faq_hits, graph_hits, expanded_graph_info):
    faq_section = "\n".join([
        f"{i+1}. Q: {f['question']}\n   A: {f['answer']}"
        for i, f in enumerate(faq_hits)
    ]) or "None"

    graph_section = "\n".join([
        f"{i+1}. Node: {g['node_name']} (type={g['node_type']}), neighbors={g.get('neighbors', [])}"
        for i, g in enumerate(graph_hits)
    ]) or "None"

    expansion_section = "\n".join([
        f"{i+1}. {node} → {neighbors}"
        for i, (node, neighbors) in enumerate(expanded_graph_info.items())
    ]) or "None"

    prompt = f"""
You are **Zubair**, a graduate student known for being polite, proactive, and clear in communication.

Your job is to draft a short, natural, and professional email reply.

Keep it warm but not overly formal — think of how a thoughtful student would respond to a professor, coordinator, or peer.

---

✉️ **Incoming Email**
\"\"\"{email_text}\"\"\"

🎯 **Detected Intent**: {intent}

📘 **Relevant FAQs**
{faq_section}

🧩 **Graph Context**
{graph_section}

🔗 **Related Concepts**
{expansion_section}

---

Write your reply in Zubair’s tone:
- Acknowledge the sender and context.
- If an action is requested, confirm or ask a polite follow-up question.
- Keep the reply under 120 words.
- Do NOT invent facts — only use what’s in context.
"""
    return prompt


In [15]:
# FIXED VERSION: answer_email_enhanced with separate FAQ search
# This ensures FAQs are always retrieved

def answer_email_enhanced_fixed(email_text: str, top_k: int = 6, show_context: bool = True):
    """
    Enhanced email answering with:
    1. Multi-intent classification
    2. Separate FAQ search (Qdrant) - ALWAYS retrieves FAQs
    3. Intent-based graph retrieval (NetworkX) - NO Qdrant for graph nodes
    4. Graph expansion
    """
    
    # Step 1: Multi-intent classification
    intents = classify_multi_intent(email_text, list(unique_intents))
    primary_intent = intents[0] if intents else "general_inquiry"
    
    if show_context:
        print(f"🎯 Detected Intents: {intents}")
        print(f"   Primary: {primary_intent}\n")
    
    # Step 2: Embed query for vector search
    q_vec = embedder.encode([email_text])[0].tolist()

    # Step 3: Search Qdrant for FAQs ONLY (separate search to ensure FAQs are always retrieved)
    try:
        from qdrant_client.models import Filter, FieldCondition, MatchValue
        faq_search_results = qdrant.query_points(
            collection_name="knowledge_space",
            query=q_vec,
            limit=top_k,
            query_filter=Filter(
                must=[FieldCondition(key="type", match=MatchValue(value="faq"))]
            )
        ).points
    except (AttributeError, TypeError, ImportError):
        # Fallback: search all and filter manually
        all_hits = qdrant.search(
            collection_name="knowledge_space",
            query_vector=q_vec,
            limit=top_k * 2,  # Get more to filter
            with_payload=True
        )
        faq_search_results = [h for h in all_hits if h.payload.get("type") == "faq"][:top_k]
    
    # Extract FAQ hits
    faq_hits = []
    for h in faq_search_results:
        p = h.payload
        if p.get("type") == "faq":
            faq_hits.append({"score": h.score, **p})

    if show_context:
        print(f"📊 FAQ Search Results:")
        print(f"   FAQ hits: {len(faq_hits)}\n")

    # Step 4: Intent-based graph retrieval (using NetworkX, NO Qdrant for graph nodes)
    intent_graph_nodes = get_nodes_by_intents(intents, limit=5)
    
    # Convert to graph hit format
    graph_hits = []
    for node in intent_graph_nodes:
        graph_hits.append({
            "score": 0.75,  # Intent-based matches get fixed score
            "node_name": node["name"],
            "node_type": node["type"],
            "neighbors": node["neighbors"],
            "relationships": node.get("relationships", {"outgoing": [], "incoming": []})
        })
    
    if show_context:
        print(f"🕸️ Intent-based Graph Retrieval:")
        print(f"   Total graph nodes: {len(graph_hits)}\n")

    # Step 5: Graph expansion
    expanded_graph_info = {}
    for g in graph_hits:
        node = g["node_name"]
        if node in G:
            neighbors = list(G.successors(node)) + list(G.predecessors(node))
            expanded_graph_info[node] = neighbors

    # Print detailed context
    if show_context:
        print("\n" + "="*70)
        print("🔍 RETRIEVED CONTEXT")
        print("="*70)
        
        print("\n📚 FAQ Chunks:")
        if faq_hits:
            for i, f in enumerate(faq_hits, 1):
                print(f"  {i}. [Score {f['score']:.3f}] Q: {f['question']}")
                print(f"     A: {f['answer'][:80]}...\n")
        else:
            print("  None\n")

        print("🕸️ Graph Nodes:")
        if graph_hits:
            for i, g in enumerate(graph_hits, 1):
                print(f"  {i}. [Score {g['score']:.3f}] {g['node_name']} (type: {g['node_type']})")
                neighbors_str = ", ".join(g.get('neighbors', [])[:5])
                print(f"     Neighbors: {neighbors_str}\n")
        else:
            print("  None\n")
        
        print("🔗 Expanded Graph Context:")
        if expanded_graph_info:
            for node, neighbors in list(expanded_graph_info.items())[:5]:
                print(f"  {node} → {', '.join(neighbors[:5])}")
        else:
            print("  None")
        
        print("="*70 + "\n")

    # Step 6: Build enhanced prompt
    prompt = build_prompt(
        email_text, 
        ", ".join(intents),
        faq_hits, 
        graph_hits, 
        expanded_graph_info
    )

    # Step 7: Generate reply
    resp = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    reply_text = resp["message"]["content"]

    # Step 8: Confidence scoring (use FAQ top score if available)
    top_score = faq_hits[0]["score"] if faq_hits else (graph_hits[0]["score"] if graph_hits else 0.0)
    auto_send = top_score > 0.85

    return {
        "intents": intents,
        "primary_intent": primary_intent,
        "reply": reply_text,
        "top_score": top_score,
        "auto_send": auto_send,
        "faq_used": faq_hits,
        "graph_used": graph_hits,
        "graph_expansion": expanded_graph_info,
    }

print("✅ answer_email_enhanced_fixed loaded!")
print("📝 Usage: result = answer_email_enhanced_fixed(email_text)")
print("🆕 Features:")
print("   - Separate FAQ search (always retrieves FAQs)")
print("   - Intent-based graph retrieval (NetworkX only)")
print("   - Both FAQs and graph nodes guaranteed to work")


✅ answer_email_enhanced_fixed loaded!
📝 Usage: result = answer_email_enhanced_fixed(email_text)
🆕 Features:
   - Separate FAQ search (always retrieves FAQs)
   - Intent-based graph retrieval (NetworkX only)
   - Both FAQs and graph nodes guaranteed to work


In [13]:
"""
Enhanced answer_email function with multi-intent classification and graph retrieval.
"""

def answer_email_enhanced(email_text: str, top_k: int = 6, show_context: bool = True):
    """
    Enhanced email answering with:
    1. Multi-intent classification
    2. Vector search (FAQs)
    3. Intent-based graph retrieval
    4. Graph expansion
    """
    
    # Step 1: Multi-intent classification
    intents = classify_multi_intent(email_text, list(unique_intents))
    primary_intent = intents[0] if intents else "general_inquiry"
    
    if show_context:
        print(f"🎯 Detected Intents: {intents}")
        print(f"   Primary: {primary_intent}\n")
    
    # Step 2: Embed query for vector search
    q_vec = embedder.encode([email_text])[0].tolist()

    # Step 3: Vector search in Qdrant
    try:
        hits = qdrant.query_points(
            collection_name="knowledge_space",
            query=q_vec,   
            limit=top_k
        ).points
    except AttributeError:
        hits = qdrant.search(
            collection_name="knowledge_space",
            query_vector=q_vec,
            limit=top_k,
            with_payload=True
        )

    # Step 4: Separate FAQ and graph hits from vector search
    faq_hits, graph_hits = [], []
    for h in hits:
        p = h.payload
        if p["type"] == "faq":
            faq_hits.append({"score": h.score, **p})
        elif p["type"] == "graph_node":
            graph_hits.append({"score": h.score, **p})

    if show_context:
        print(f"📊 Vector Search Results:")
        print(f"   FAQ hits: {len(faq_hits)}")
        print(f"   Graph hits from vector search: {len(graph_hits)}\n")

    # Step 5: Intent-based graph retrieval (NEW!)
    intent_graph_nodes = get_nodes_by_intents(intents, limit=5)
    
    # Convert to graph hit format and merge with vector search results
    for node in intent_graph_nodes:
        # Avoid duplicates
        if not any(g.get("node_name") == node["name"] for g in graph_hits):
            graph_hits.append({
                "score": 0.75,  # Assign reasonable score for intent-based matches
                "node_name": node["name"],
                "node_type": node["type"],
                "neighbors": node["neighbors"]
            })
    
    if show_context:
        print(f"🕸️ After Intent-based Graph Retrieval:")
        print(f"   Total graph nodes: {len(graph_hits)}\n")

    # Step 6: Graph expansion
    expanded_graph_info = {}
    for g in graph_hits:
        node = g["node_name"]
        if node in G:
            neighbors = list(G.successors(node)) + list(G.predecessors(node))
            expanded_graph_info[node] = neighbors

    # Print detailed context
    if show_context:
        print("\n" + "="*70)
        print("🔍 RETRIEVED CONTEXT")
        print("="*70)
        
        print("\n📚 FAQ Chunks:")
        if faq_hits:
            for i, f in enumerate(faq_hits, 1):
                print(f"  {i}. [Score {f['score']:.3f}] Q: {f['question']}")
                print(f"     A: {f['answer'][:80]}...\n")
        else:
            print("  None\n")

        print("🕸️ Graph Nodes:")
        if graph_hits:
            for i, g in enumerate(graph_hits, 1):
                print(f"  {i}. [Score {g['score']:.3f}] {g['node_name']} (type: {g['node_type']})")
                neighbors_str = ", ".join(g.get('neighbors', [])[:5])
                print(f"     Neighbors: {neighbors_str}\n")
        else:
            print("  None\n")
        
        print("🔗 Expanded Graph Context:")
        if expanded_graph_info:
            for node, neighbors in list(expanded_graph_info.items())[:5]:
                print(f"  {node} → {', '.join(neighbors[:5])}")
        else:
            print("  None")
        
        print("="*70 + "\n")

    # Step 7: Build enhanced prompt
    prompt = build_prompt(
        email_text, 
        ", ".join(intents),  # Pass all intents
        faq_hits, 
        graph_hits, 
        expanded_graph_info
    )

    # Step 8: Generate reply
    resp = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    reply_text = resp["message"]["content"]

    # Step 9: Confidence scoring
    top_score = hits[0].score if hits else 0.0
    auto_send = top_score > 0.85

    return {
        "intents": intents,
        "primary_intent": primary_intent,
        "reply": reply_text,
        "top_score": top_score,
        "auto_send": auto_send,
        "faq_used": faq_hits,
        "graph_used": graph_hits,
        "graph_expansion": expanded_graph_info,
    }


def classify_multi_intent(email_text: str, available_intents: list) -> list:
    """
    Classify multiple intents in an email using LLM.
    Returns list of intents (can be multiple).
    """
    intents_str = ", ".join(available_intents)
    
    prompt = f"""You are an email intent classifier.

Available intents (choose from these EXACT labels):
{intents_str}

Email to classify:
\"\"\"{email_text}\"\"\"

Instructions:
1. Identify ALL relevant intents from the list above
2. An email can have multiple intents (e.g., "share resume" = send_materials, "schedule meeting" = schedule)
3. Return ONLY a JSON array with exact labels from the list
4. Use underscores for multi-word intents (e.g., send_materials, not "send materials")

Examples:
- "Can you send me your resume?" → ["send_materials"]
- "Share your resume and let's schedule a call" → ["send_materials", "schedule"]
- "What time works for a meeting?" → ["schedule"]

Return ONLY valid JSON array, nothing else:
"""
    
    try:
        response = ollama.chat(
            model="llama3",
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response["message"]["content"].strip()
        
        # Debug: print what LLM returned
        print(f"🔍 LLM raw response: {content[:200]}...")
        
        # Try to extract JSON from markdown code blocks
        import json
        import re
        
        # Remove markdown code blocks if present
        content = re.sub(r'\s*', '', content)
        content = re.sub(r'```\s*', '', content)
        content = content.strip()
        
        try:
            intents = json.loads(content)
            if isinstance(intents, list):
                # Normalize intents
                normalized = []
                known_lower = [i.lower().replace(" ", "_") for i in available_intents]
                
                for intent in intents:
                    intent_clean = str(intent).lower().replace(" ", "_").strip()
                    # Try to find match (exact or partial)
                    if intent_clean in known_lower:
                        # Get the original case from available_intents
                        idx = known_lower.index(intent_clean)
                        normalized.append(available_intents[idx])
                    else:
                        # Try fuzzy match
                        for avail in available_intents:
                            if intent_clean == avail.lower().replace(" ", "_"):
                                normalized.append(avail)
                                break
                
                if normalized:
                    print(f"✅ Classified intents: {normalized}")
                    return normalized
                else:
                    print(f"⚠️ No valid intents found. LLM returned: {intents}")
        except json.JSONDecodeError as e:
            print(f"⚠️ JSON parse error: {e}")
            print(f"   Raw content: {content[:200]}")
            # Try to extract intent names from text
            for intent in available_intents:
                if intent.lower() in content.lower():
                    print(f"✅ Found intent in text: {intent}")
                    return [intent]
        
        print(f"⚠️ Defaulting to general_inquiry")
        return ["general_inquiry"]
        
    except Exception as e:
        print(f"⚠️ Intent classification error: {e}")
        return ["general_inquiry"]## Key Improvements

def get_nodes_by_intents(intents: list, limit: int = 5) -> list:
    """
    Retrieve graph nodes related to the detected intents.
    This is the key enhancement - direct intent-to-node lookup!
    INCLUDES RELATIONSHIPS from edges!
    """
    def get_relationships(node_name: str):
        """Extract relationship information from edges."""
        if node_name not in G:
            return {"outgoing": [], "incoming": []}
        
        outgoing = []
        for successor in G.successors(node_name):
            edge_data = G.get_edge_data(node_name, successor, {})
            outgoing.append({
                "node": successor,
                "relation": edge_data.get("relation", "CONNECTED"),
                "email_id": edge_data.get("email_id")
            })
        
        incoming = []
        for predecessor in G.predecessors(node_name):
            edge_data = G.get_edge_data(predecessor, node_name, {})
            incoming.append({
                "node": predecessor,
                "relation": edge_data.get("relation", "CONNECTED"),
                "email_id": edge_data.get("email_id")
            })
        
        return {"outgoing": outgoing, "incoming": incoming}
    
    nodes = []
    seen_names = set()
    
    for intent in intents:
        # Check if intent exists as a node in graph
        if intent in G:
            # Get node info
            node_data = G.nodes[intent]
            neighbors = list(G.successors(intent)) + list(G.predecessors(intent))
            relationships = get_relationships(intent)  # ← Get relationships!
            
            if intent not in seen_names:
                nodes.append({
                    "name": intent,
                    "type": node_data.get("type", "unknown"),
                    "neighbors": neighbors,
                    "relationships": relationships  # ← Include relationships!
                })
                seen_names.add(intent)
            
            # Also get connected nodes (topics, artifacts)
            for neighbor in neighbors:
                if neighbor not in seen_names and len(nodes) < limit:
                    neighbor_data = G.nodes[neighbor]
                    neighbor_neighbors = list(G.successors(neighbor)) + list(G.predecessors(neighbor))
                    neighbor_relationships = get_relationships(neighbor)  # ← Get relationships!
                    nodes.append({
                        "name": neighbor,
                        "type": neighbor_data.get("type", "unknown"),
                        "neighbors": neighbor_neighbors,
                        "relationships": neighbor_relationships  # ← Include relationships!
                    })
                    seen_names.add(neighbor)
    
    return nodes[:limit]


print("✅ Enhanced answer_email function loaded!")
print("📝 Usage: result = answer_email_enhanced(email_text)")
print("🆕 New features:")
print("   - Multi-intent classification")
print("   - Intent-based graph retrieval")
print("   - Enhanced context display")



✅ Enhanced answer_email function loaded!
📝 Usage: result = answer_email_enhanced(email_text)
🆕 New features:
   - Multi-intent classification
   - Intent-based graph retrieval
   - Enhanced context display


In [16]:
test_email = """Hi Zubair,
Thank you for reaching out. To proceed with your interest in the Advanced Analytics position at Google, kindly share your resume and provide the right time to connect with you for an online meeting.
Regards,
Arjun Das
Talent Acquisition
Google Inc."""

result = answer_email_enhanced_fixed(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🔍 LLM raw response: ["send_materials", "request_info", "schedule"]...
✅ Classified intents: ['send_materials', 'request_info', 'schedule']
🎯 Detected Intents: ['send_materials', 'request_info', 'schedule']
   Primary: send_materials

📊 FAQ Search Results:
   FAQ hits: 2

🕸️ Intent-based Graph Retrieval:
   Total graph nodes: 5


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.375] Q: Can you share your resume and portfolio?
     A: Sure, here are the links to my resume and portfolio: Resume - https://drive.goog...

  2. [Score 0.373] Q: What additional information has been provided by Zubair for his application?
     A: Zubair has attached previous evaluation forms and feedback reports, a summary of...

🕸️ Graph Nodes:
  1. [Score 0.750] send_materials (type: intent)
     Neighbors: Group/Event Coordination, Professor/Academic, Scheduling, Recruiter/Job Search

  2. [Score 0.750] Group/Event Coordination (type: topic)
     Neighbors: accept_or_decline, reschedule, request_info, send_m

### The top similarity score measures how semantically close your input is to your best-matching knowledge chunk. It’s a confidence proxy for deciding whether the LLM’s reply is likely grounded in the right retrieved context.

In [18]:
test_email = (
    "Hi Zubair, I hope you’re doing well. Can I get your linkedin profile to connect?"
)
result = answer_email_enhanced_fixed(test_email)

result = answer_email_enhanced_fixed(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🔍 LLM raw response: ["request_info"]...
✅ Classified intents: ['request_info']
🎯 Detected Intents: ['request_info']
   Primary: request_info

📊 FAQ Search Results:
   FAQ hits: 3

🕸️ Intent-based Graph Retrieval:
   Total graph nodes: 5


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.668] Q: Where can I find your LinkedIn profile?
     A: You can view my LinkedIn profile at https://www.linkedin.com/in/zubair-atha/...

  2. [Score 0.549] Q: Where can I find Zubair's professional profiles?
     A: Zubair maintains a LinkedIn profile (https://www.linkedin.com/in/zubair-atha/) a...

  3. [Score 0.481] Q: What additional information has been provided by Zubair for his application?
     A: Zubair has attached previous evaluation forms and feedback reports, a summary of...

🕸️ Graph Nodes:
  1. [Score 0.750] request_info (type: intent)
     Neighbors: Group/Event Coordination, Professor/Academic, Feedback & Reviews, Scheduling, Recruiter/Job Search

  2. [Score 0.750] Group/Event Coordinat